In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import re
import sys
import shutil
import zipfile
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm
from urllib.parse import urlparse, parse_qs
from tqdm.notebook import tqdm
sys.path.append(r'E:\repository\dataset_tools\isds_tool\PS_data')
from vis import select_defect, esresult2yolo, esimage_merge
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [6]:
root_dir = r"Y:\ZHL\isds\PS"
RESULT_ALL = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportResults?subProjectId="
RESULT_JSON = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportOutputResults?subProjectId="
CONFIG_EXPORT = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/getSubprojectMetaData?subProjectId="
anno_dir = r'E:\data\202502_signboard\data_annotation\ps_data\task\merge_dir_0811'


subproject_list = [
    # 301, 303, 304, 310, 306, 308, 309
    # 342, 343, 344, 345, 346, 347, 348, 349, 350, 352, 353, 354, 355, 356, 358, 359, 360, 361, 363, 363, 364, 365, 366, 367, 368 
    369, 370
]

merge_dir = r'Y:\ZHL\isds\PS\results\task_0811'

In [10]:
# def extract_date_from_project_name(project_name):
#     pattern = r"^2025(\d{4})_SIT$"
#     match = re.match(pattern, project_name)
    
#     if match:
#         date = match.group(1)  # 提取 xxxx
#         return date
#     else:
#         return 0000

def extract_date_from_project_name(project_name):
    pattern = r"^2025(\d{4})_UAT$"
    match = re.match(pattern, project_name)
    
    if match:
        date = match.group(1)  # 提取 xxxx
        return date
    else:
        return 0000

In [11]:
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

# 多线程下载函数
def download_zip_files_mp(subproject_list, root_dir, overwirte=False, max_workers=4):
    zip_download_dict = {}

    def download_single(subproject_id):
        # 每个线程创建独立的session
        session = requests.Session()
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        })

        download_url = RESULT_ALL+str(subproject_id)
        config_url = CONFIG_EXPORT+str(subproject_id)

        try:
            print(f"🔗 正在连接: {config_url}")
            response = session.get(config_url)
            response.raise_for_status()
            data = response.json()
            project_name = data['project']['name']
            date = extract_date_from_project_name(project_name)

            save_dir = os.path.join(root_dir, 'task'+date, 'results')
            os.makedirs(save_dir, exist_ok=True)

            print(f"🔗 正在连接: {download_url}")
            response = session.get(download_url, stream=True)
            response.raise_for_status()

            total_size = int(response.headers.get("content-length", 0))
            chunk_size = 1024 * 1024  # 1MB

            print(f'total size: {total_size/chunk_size} MB')

            parsed_url = urlparse(download_url)
            query_params = parse_qs(parsed_url.query)
            sub_project_id = query_params.get('subProjectId', ['unknown'])[0]
            filename = f"{sub_project_id}.zip"
            save_path = os.path.join(save_dir, filename)

            if not os.path.exists(save_path) or overwirte:
                with open(save_path, 'wb') as f, tqdm(
                    total=total_size,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"📥 下载 {filename} -> {save_dir}",
                    leave=True,
                ) as pbar:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

                print(f"✅ 下载完成: {save_path}\n")
                return subproject_id, save_path
            else:
                print(f"⚠️ 文件已存在，跳过下载: {save_path}\n")
                return subproject_id, save_path
        except Exception as e:
            print(f"❌ 下载失败: {download_url}\n原因: {e}")
            return subproject_id, None

    # 使用线程池执行下载任务
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        futures = {executor.submit(download_single, sid): sid for sid in subproject_list}

        # 收集结果
        for future in concurrent.futures.as_completed(futures):
            subproject_id, save_path = future.result()
            if save_path:
                zip_download_dict[subproject_id] = save_path

    return zip_download_dict

In [12]:
# zip_download_dict = download_zip_files(subproject_list, root_dir)
zip_download_dict = download_zip_files_mp(subproject_list, root_dir)

🔗 正在连接: http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/getSubprojectMetaData?subProjectId=369
🔗 正在连接: http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/getSubprojectMetaData?subProjectId=370
🔗 正在连接: http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportResults?subProjectId=369
🔗 正在连接: http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportResults?subProjectId=370
total size: 0.0 MBtotal size: 0.0 MB



📥 下载 369.zip -> Y:\ZHL\isds\PS\task0812\results: 0.00B [00:00, ?B/s]

📥 下载 370.zip -> Y:\ZHL\isds\PS\task0812\results: 0.00B [00:00, ?B/s]

✅ 下载完成: Y:\ZHL\isds\PS\task0812\results\370.zip

✅ 下载完成: Y:\ZHL\isds\PS\task0812\results\369.zip



In [13]:
def simple_unzip(zip_path, dst_dir):
    print(f'{zip_path} unzip...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dst_dir)
    print(f'{zip_path} done\n')

In [14]:
for key, zip_path in zip_download_dict.items():
    simple_unzip(zip_path, dst_dir=zip_path.replace('.zip', ''))


Y:\ZHL\isds\PS\task0812\results\370.zip unzip...
Y:\ZHL\isds\PS\task0812\results\370.zip done

Y:\ZHL\isds\PS\task0812\results\369.zip unzip...
Y:\ZHL\isds\PS\task0812\results\369.zip done



In [14]:
for key, zip_path in zip_download_dict.items():
    esimage_merge(zip_path.replace('.zip', ''))

reset Y:\ZHL\isds\PS\task0722\results\345\yolo_dataset\images...


100%|██████████| 864/864 [07:51<00:00,  1.83it/s]


reset Y:\ZHL\isds\PS\task0722\results\342\yolo_dataset\images...


100%|██████████| 1074/1074 [11:55<00:00,  1.50it/s]


reset Y:\ZHL\isds\PS\task0722\results\344\yolo_dataset\images...


...

100%|██████████| 330/330 [03:43<00:00,  1.47it/s]


reset Y:\ZHL\isds\PS\task0808\results\368\yolo_dataset\images...


100%|██████████| 94/94 [00:58<00:00,  1.61it/s]


reset Y:\ZHL\isds\PS\task0808\results\366\yolo_dataset\images...


100%|██████████| 256/256 [02:47<00:00,  1.53it/s]


In [18]:
for key, zip_path in zip_download_dict.items():
    yolo_dir = os.path.join(zip_path.replace('.zip', ''), 'yolo_dataset')
    img_dir = os.path.join(yolo_dir, 'images')
    
    print(f'{img_dir} selecting...')
    image_dir_select = img_dir+'_select'
    if os.path.exists(image_dir_select):
        continue
    shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
    select_img(img_dir, image_dir_select, gap=10)
    if len(os.listdir(image_dir_select)) == 0:
        continue

    print(f'{image_dir_select} filtering...')
    image_dir_filter = img_dir+'_filter'
    shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
    filter_deduplication(image_dir_select, image_dir_filter)

    print(f'process {img_dir}, {len(os.listdir(img_dir))} -> {len(os.listdir(image_dir_select))} -> {len(os.listdir(image_dir_filter))}')

Y:\ZHL\isds\PS\task0722\results\345\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0722\results\342\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0722\results\344\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0722\results\343\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0725\results\346\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0730\results\348\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0730\results\349\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0725\results\347\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0801\results\354\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0801\results\352\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0801\results\356\yolo_dataset\images selecting...
Y:\ZHL\isds\PS\task0730\results\350\yolo_dataset\images selecting...


100%|██████████| 98/98 [00:13<00:00,  7.07it/s]


Y:\ZHL\isds\PS\task0730\results\350\yolo_dataset\images_select filtering...


100%|██████████| 98/98 [00:07<00:00, 12.81it/s]



Total unique images copied: 98
process Y:\ZHL\isds\PS\task0730\results\350\yolo_dataset\images, 972 -> 98 -> 98
Y:\ZHL\isds\PS\task0801\results\353\yolo_dataset\images selecting...


...


Total unique images copied: 57
process Y:\ZHL\isds\PS\task0808\results\368\yolo_dataset\images, 564 -> 57 -> 57
Y:\ZHL\isds\PS\task0808\results\366\yolo_dataset\images selecting...


100%|██████████| 154/154 [00:12<00:00, 12.80it/s]


Y:\ZHL\isds\PS\task0808\results\366\yolo_dataset\images_select filtering...


100%|██████████| 154/154 [00:06<00:00, 23.65it/s]


Total unique images copied: 154
process Y:\ZHL\isds\PS\task0808\results\366\yolo_dataset\images, 1536 -> 154 -> 154


In [19]:
for key, zip_path in zip_download_dict.items():
    esresult2yolo(zip_path.replace('.zip', ''))

reset Y:\ZHL\isds\PS\task0722\results\345\yolo_dataset\labels...


100%|██████████| 864/864 [03:26<00:00,  4.18it/s]


reset Y:\ZHL\isds\PS\task0722\results\342\yolo_dataset\labels...


100%|██████████| 1074/1074 [04:26<00:00,  4.03it/s]


reset Y:\ZHL\isds\PS\task0722\results\344\yolo_dataset\labels...


...

100%|██████████| 330/330 [00:42<00:00,  7.69it/s]


reset Y:\ZHL\isds\PS\task0808\results\368\yolo_dataset\labels...


100%|██████████| 94/94 [00:20<00:00,  4.58it/s]


reset Y:\ZHL\isds\PS\task0808\results\366\yolo_dataset\labels...


100%|██████████| 256/256 [00:56<00:00,  4.51it/s]


In [20]:
for key, zip_path in zip_download_dict.items():
    yolo_dir = os.path.join(zip_path.replace('.zip', ''), 'yolo_dataset')
    img_dir = os.path.join(yolo_dir, 'images')

    select_defect(yolo_dir, yolo_dir+'_defect', anno_dir)

100%|██████████| 5184/5184 [00:54<00:00, 95.19it/s] 


select 144 from 519


100%|██████████| 6444/6444 [00:44<00:00, 143.89it/s]


select 83 from 645


100%|██████████| 5724/5724 [00:42<00:00, 134.92it/s]


...

select 1 from 198


100%|██████████| 564/564 [00:05<00:00, 109.57it/s]


select 4 from 57


100%|██████████| 1536/1536 [00:12<00:00, 126.28it/s]

select 7 from 154


In [26]:

merge_img_dir = os.path.join(merge_dir, 'images')
merge_label_dir = os.path.join(merge_dir, 'labels')

os.makedirs(merge_img_dir, exist_ok=True)
os.makedirs(merge_label_dir, exist_ok=True)

for key, zip_path in zip_download_dict.items():
    yolo_dir = os.path.join(zip_path.replace('.zip', ''), 'yolo_dataset_defect')
    img_dir = os.path.join(yolo_dir, 'images')
    label_dir = os.path.join(yolo_dir, 'labels')

    # 合并 images 目录
    print(f'processing {img_dir}')
    shutil.copytree(img_dir, merge_img_dir, dirs_exist_ok=True)
    # 合并 labels 目录
    print(f'processing {label_dir}')
    shutil.copytree(label_dir, merge_label_dir, dirs_exist_ok=True)
    


processing Y:\ZHL\isds\PS\task0722\results\345\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0722\results\345\yolo_dataset_defect\labels
processing Y:\ZHL\isds\PS\task0722\results\342\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0722\results\342\yolo_dataset_defect\labels
processing Y:\ZHL\isds\PS\task0722\results\344\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0722\results\344\yolo_dataset_defect\labels
processing Y:\ZHL\isds\PS\task0722\results\343\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0722\results\343\yolo_dataset_defect\labels
processing Y:\ZHL\isds\PS\task0725\results\346\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0725\results\346\yolo_dataset_defect\labels
processing Y:\ZHL\isds\PS\task0730\results\348\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0730\results\348\yolo_dataset_defect\labels
processing Y:\ZHL\isds\PS\task0730\results\349\yolo_dataset_defect\images
processing Y:\ZHL\isds\PS\task0730\res

In [27]:
def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

In [28]:
zip_folder_to_path(merge_img_dir, merge_dir+'_image.zip')
zip_folder_to_path(merge_label_dir, merge_dir+'_label.zip')


zip 'Y:\ZHL\isds\PS\results\task_0811\images' to 'Y:\ZHL\isds\PS\results\task_0811_image.zip'
zip 'Y:\ZHL\isds\PS\results\task_0811\labels' to 'Y:\ZHL\isds\PS\results\task_0811_label.zip'
